# MIST-HER2: multi-model image and error audit

This notebook compares full-test predictions from multiple models against the same 1053 MIST
pairs. It also inventories small qualitative experiment galleries without mixing them into the
quantitative benchmark. The audit is diagnostic: it does not replace the unchanged full-test
metrics and does not assign clinical HER2 classes.

**Kaggle accelerator:** CPU is sufficient. Attach the public MIST-HER2 dataset, the baseline-256
prediction ZIP, the VP-CFM output or ZIP, and the experiment collection ZIP.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import shutil
import stat
import tempfile
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.ndimage import laplace, shift as ndi_shift, sobel
from scipy.stats import spearmanr, wilcoxon
from skimage.color import rgb2gray, rgb2hed
from skimage.metrics import structural_similarity
from skimage.registration import phase_cross_correlation

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
OUTPUT = WORK_ROOT / "mist-multimodel-error-audit"
EXTRACTED = WORK_ROOT / "extracted-audit-inputs"
for path in (OUTPUT, EXTRACTED):
    path.mkdir(parents=True, exist_ok=True)

DATASET_OVERRIDE = None
MODEL_OVERRIDES = {}  # Example: {"baseline_256": "/kaggle/input/.../predictions"}
AUDIT_SIZE = 256
DAB_THRESHOLD = 0.05
BOOTSTRAP_REPEATS = 2000
GALLERY_SIZE = 6
SEED = 42


## 1. Safely extract uploaded archives and locate MIST


In [ ]:
def safe_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / ".complete"
    if marker.exists():
        return
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"Unsafe archive path: {member.filename}")
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Archive symlinks are unsupported")
        archive.extractall(destination)
    marker.write_text("ok")


archives = sorted(INPUT_ROOT.rglob("*.zip"))
for archive in archives:
    key = hashlib.sha256(str(archive).encode()).hexdigest()[:10]
    destination = EXTRACTED / f"{archive.stem}-{key}"
    try:
        safe_extract(archive, destination)
        print("Extracted:", archive.name, "->", destination)
    except zipfile.BadZipFile:
        print("Skipped non-ZIP payload:", archive)


SEARCH_ROOTS = [INPUT_ROOT, EXTRACTED]

def locate_dataset():
    if DATASET_OVERRIDE:
        return Path(DATASET_OVERRIDE)
    markers = []
    for root in SEARCH_ROOTS:
        markers.extend(root.rglob("kaggle_bundle.json"))
    candidates = []
    for marker in {p.resolve(): p for p in markers}.values():
        bundle = marker.parent
        manifest = bundle / "manifests" / "group-disjoint" / "pairs.jsonl"
        data_root = bundle / "data" / "HER2"
        if manifest.is_file() and data_root.is_dir():
            candidates.append(bundle)
    if not candidates:
        raise FileNotFoundError("No complete MIST bundle was found")
    candidates.sort(key=lambda p: ("mist-her2" not in str(p).lower(), len(str(p))))
    if len(candidates) > 1:
        print("Multiple MIST copies found; using:", candidates[0])
    return candidates[0]


BUNDLE = locate_dataset()
DATA_ROOT = BUNDLE / "data" / "HER2"
MANIFEST = BUNDLE / "manifests" / "group-disjoint" / "pairs.jsonl"
records = [json.loads(line) for line in MANIFEST.read_text().splitlines()
           if line.strip() and json.loads(line)["split"] == "test"]
if len(records) != 1053:
    raise RuntimeError(f"Expected 1053 test records, found {len(records)}")
print("Dataset:", BUNDLE)
print("Test records:", len(records))


## 2. Discover complete prediction sets

A directory is accepted as a model only when it contains at least 95% of the exact manifest IDs.
Small `example-*.png` galleries are therefore never mistaken for full predictions.


In [ ]:
expected_relpaths = [Path(f"{record['id']}.png") for record in records]

def candidate_roots():
    roots = set()
    for search_root in SEARCH_ROOTS:
        for png in search_root.rglob("*.png"):
            if png.parent.name in {"train", "test"}:
                roots.add(png.parent.parent.resolve())
    return sorted(roots)


def sampling_metadata(root):
    candidates = [root / "sampling.json", root.parent / "sampling.json"]
    for path in candidates:
        if path.is_file():
            try:
                return json.loads(path.read_text())
            except Exception:
                pass
    return {}


def infer_label(root, metadata):
    text = (str(root) + " " + json.dumps(metadata)).lower()
    flow_variant = str(metadata.get("flow_variant", "")).lower()
    if "variance" in text or "trigonometric" in flow_variant:
        return "VP-CFM 512"
    if "exact" in text and "ot" in text:
        return "Exact OT-CFM 512"
    if "256" in text or metadata.get("crop_size") == 256:
        return "CFM baseline 256"
    return root.name


models = {}
inventory = []
for root in candidate_roots():
    present = sum((root / rel).is_file() for rel in expected_relpaths)
    coverage = present / len(expected_relpaths)
    metadata = sampling_metadata(root)
    inventory.append({"root": str(root), "present": present, "coverage": coverage,
                      "metadata": json.dumps(metadata, sort_keys=True)})
    if coverage >= 0.95:
        label = infer_label(root, metadata)
        base = label
        suffix = 2
        while label in models:
            label = f"{base} ({suffix})"
            suffix += 1
        models[label] = root

for label, path in MODEL_OVERRIDES.items():
    models[label] = Path(path)

display(pd.DataFrame(inventory).sort_values("coverage", ascending=False).head(20))
print("Accepted full prediction sets:")
for label, root in models.items():
    print(" ", label, "->", root)
if len(models) < 2:
    raise RuntimeError("At least two complete prediction sets are required; inspect the inventory or set MODEL_OVERRIDES")


## 3. Per-image measurements

The audit measures paired reconstruction, stain amount, sharpness, morphology proxies and pair
alignment. Alignment is estimated only for stratification; it is not used to alter test images.


In [ ]:
def read_rgb(path, size=AUDIT_SIZE):
    with Image.open(path) as image:
        image = image.convert("RGB")
        if size is not None:
            image = image.resize((size, size), Image.Resampling.LANCZOS)
        return np.asarray(image, dtype=np.float32) / 255.0


def tissue_mask(rgb):
    gray = rgb2gray(rgb)
    chroma = rgb.max(axis=-1) - rgb.min(axis=-1)
    return (gray < 0.90) | ((gray < 0.96) & (chroma > 0.06))


def robust01(array):
    low, high = np.percentile(array, [1, 99])
    return np.clip((array - low) / max(high - low, 1e-6), 0, 1)


def nuclear_map(rgb):
    return robust01(rgb2hed(np.clip(rgb, 1e-6, 1))[..., 0])


def dab_map(rgb):
    return np.maximum(rgb2hed(np.clip(rgb, 1e-6, 1))[..., 2], 0)


def safe_corr(first, second, mask=None):
    if mask is not None:
        first, second = first[mask], second[mask]
    first, second = first.ravel(), second.ravel()
    if len(first) < 32 or np.std(first) < 1e-7 or np.std(second) < 1e-7:
        return np.nan
    return float(np.corrcoef(first, second)[0, 1])


def edge_map(array):
    return np.hypot(sobel(array, axis=0), sobel(array, axis=1))


def pair_features(record):
    he = read_rgb(DATA_ROOT / record["he"])
    real = read_rgb(DATA_ROOT / record["ihc"])
    he_nuclear, real_nuclear = nuclear_map(he), nuclear_map(real)
    shift, _, _ = phase_cross_correlation(he_nuclear, real_nuclear, upsample_factor=2)
    aligned_real = ndi_shift(real_nuclear, shift=shift, order=1, mode="nearest")
    mask = tissue_mask(he) & tissue_mask(real)
    real_dab = dab_map(real)
    return he, real, {
        "id": record["id"], "group": record["group"],
        "he_tissue_fraction": float(tissue_mask(he).mean()),
        "real_tissue_fraction": float(tissue_mask(real).mean()),
        "translation_dy_256": float(shift[0]), "translation_dx_256": float(shift[1]),
        "translation_magnitude_256": float(np.linalg.norm(shift)),
        "nuclear_ssim_raw": float(structural_similarity(he_nuclear, real_nuclear, data_range=1)),
        "nuclear_ssim_aligned": float(structural_similarity(he_nuclear, aligned_real, data_range=1)),
        "nuclear_edge_corr_aligned": safe_corr(edge_map(he_nuclear), edge_map(aligned_real), mask),
        "real_dab_fraction": float((real_dab > DAB_THRESHOLD).mean()),
        "real_dab_mean": float(real_dab.mean()),
        "real_sharpness": float(laplace(rgb2gray(real)).var()),
    }


def prediction_features(label, root, record, he, real, base):
    predicted = read_rgb(root / f"{record['id']}.png")
    pred_dab, real_dab = dab_map(predicted), dab_map(real)
    pred_nuclear, real_nuclear = nuclear_map(predicted), nuclear_map(real)
    mse = float(np.mean((predicted - real) ** 2))
    mae = float(np.mean(np.abs(predicted - real)))
    sharpness = float(laplace(rgb2gray(predicted)).var())
    return {
        **base, "model": label,
        "ssim": float(structural_similarity(predicted, real, data_range=1, channel_axis=-1)),
        "mse": mse, "mae": mae,
        "psnr_db": float(-10 * np.log10(max(mse, 1e-12))),
        "pred_dab_fraction": float((pred_dab > DAB_THRESHOLD).mean()),
        "dab_fraction_signed_error": float((pred_dab > DAB_THRESHOLD).mean() - (real_dab > DAB_THRESHOLD).mean()),
        "dab_fraction_abs_error": float(abs((pred_dab > DAB_THRESHOLD).mean() - (real_dab > DAB_THRESHOLD).mean())),
        "pred_dab_mean": float(pred_dab.mean()),
        "dab_mean_abs_error": float(abs(pred_dab.mean() - real_dab.mean())),
        "sharpness": sharpness,
        "sharpness_ratio": float(sharpness / max(base["real_sharpness"], 1e-8)),
        "pred_real_nuclear_ssim": float(structural_similarity(pred_nuclear, real_nuclear, data_range=1)),
        "he_pred_nuclear_ssim": float(structural_similarity(nuclear_map(he), pred_nuclear, data_range=1)),
        "pred_real_edge_corr": safe_corr(edge_map(pred_nuclear), edge_map(real_nuclear)),
    }


In [ ]:
rows = []
for index, record in enumerate(records):
    he, real, base = pair_features(record)
    for label, root in models.items():
        path = root / f"{record['id']}.png"
        if path.is_file():
            rows.append(prediction_features(label, root, record, he, real, base))
    if (index + 1) % 100 == 0 or index + 1 == len(records):
        print(f"Processed {index + 1}/{len(records)}", flush=True)

errors = pd.DataFrame(rows)
if errors.groupby("model").size().min() != len(records):
    print(errors.groupby("model").size())
    raise RuntimeError("Every accepted model must contain all test predictions")

alignment_score = errors.drop_duplicates("id").set_index("id")["nuclear_ssim_aligned"]
alignment_bins = pd.qcut(alignment_score.rank(method="first"), 4,
                         labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
dab_score = errors.drop_duplicates("id").set_index("id")["real_dab_fraction"]
dab_bins = pd.qcut(dab_score.rank(method="first"), 3, labels=["low", "medium", "high"])
errors["alignment_stratum"] = errors["id"].map(alignment_bins)
errors["dab_stratum"] = errors["id"].map(dab_bins)
errors.to_csv(OUTPUT / "per_image_errors.csv", index=False)
display(errors.head())


## 4. Aggregate results, uncertainty and paired comparisons


In [ ]:
def group_bootstrap(frame, column, repeats=BOOTSTRAP_REPEATS):
    grouped = {group: values[column].to_numpy() for group, values in frame.groupby("group")}
    groups = list(grouped)
    rng = np.random.default_rng(SEED)
    estimates = []
    for _ in range(repeats):
        sampled = rng.choice(groups, len(groups), replace=True)
        estimates.append(np.concatenate([grouped[group] for group in sampled]).mean())
    return np.percentile(estimates, [2.5, 97.5]).tolist()


metrics = ["ssim", "mae", "psnr_db", "dab_fraction_abs_error", "dab_mean_abs_error",
           "sharpness_ratio", "pred_real_nuclear_ssim", "he_pred_nuclear_ssim"]
summary_rows = []
for model, frame in errors.groupby("model"):
    row = {"model": model, "count": len(frame)}
    for metric in metrics:
        row[metric] = float(frame[metric].mean())
        if metric in {"ssim", "mae", "dab_fraction_abs_error"}:
            row[f"{metric}_group_bootstrap_95ci"] = group_bootstrap(frame, metric)
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT / "aggregate_metrics.csv", index=False)
display(summary)

paired_rows = []
labels = list(models)
for i, first in enumerate(labels):
    for second in labels[i + 1:]:
        left = errors[errors.model == first].set_index("id")
        right = errors[errors.model == second].set_index("id")
        common = left.index.intersection(right.index)
        for metric in ["ssim", "mae", "dab_fraction_abs_error", "sharpness_ratio"]:
            delta = left.loc[common, metric] - right.loc[common, metric]
            statistic, pvalue = wilcoxon(delta) if np.any(delta != 0) else (0.0, 1.0)
            paired_rows.append({"model_a": first, "model_b": second, "metric": metric,
                                "mean_a_minus_b": float(delta.mean()),
                                "median_a_minus_b": float(delta.median()),
                                "wilcoxon_p": float(pvalue)})
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT / "paired_model_comparisons.csv", index=False)
display(paired)


## 5. Error stratification and diagnostic correlations


In [ ]:
stratified = errors.groupby(["model", "alignment_stratum", "dab_stratum"], observed=True).agg(
    count=("id", "size"), ssim=("ssim", "mean"), mae=("mae", "mean"),
    dab_error=("dab_fraction_abs_error", "mean"), sharpness_ratio=("sharpness_ratio", "mean")
).reset_index()
stratified.to_csv(OUTPUT / "stratified_metrics.csv", index=False)
display(stratified)

correlation_rows = []
for model, frame in errors.groupby("model"):
    for predictor, outcome in [
        ("nuclear_ssim_aligned", "ssim"),
        ("translation_magnitude_256", "ssim"),
        ("real_dab_fraction", "dab_fraction_abs_error"),
        ("real_dab_fraction", "dab_fraction_signed_error"),
        ("he_pred_nuclear_ssim", "ssim"),
    ]:
        rho, pvalue = spearmanr(frame[predictor], frame[outcome], nan_policy="omit")
        correlation_rows.append({"model": model, "predictor": predictor, "outcome": outcome,
                                 "spearman_rho": float(rho), "p_value": float(pvalue)})
correlations = pd.DataFrame(correlation_rows)
correlations.to_csv(OUTPUT / "error_correlations.csv", index=False)
display(correlations)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
sns.boxplot(data=errors, x="alignment_stratum", y="ssim", hue="model", ax=axes[0, 0])
sns.boxplot(data=errors, x="dab_stratum", y="dab_fraction_abs_error", hue="model", ax=axes[0, 1])
sns.scatterplot(data=errors, x="real_dab_fraction", y="pred_dab_fraction", hue="model",
                alpha=0.35, s=18, ax=axes[0, 2])
sns.boxplot(data=errors, x="model", y="sharpness_ratio", ax=axes[1, 0])
sns.scatterplot(data=errors, x="nuclear_ssim_aligned", y="ssim", hue="model",
                alpha=0.35, s=18, ax=axes[1, 1])
sns.scatterplot(data=errors, x="he_pred_nuclear_ssim", y="pred_real_nuclear_ssim", hue="model",
                alpha=0.35, s=18, ax=axes[1, 2])
for axis in axes.flat:
    axis.grid(alpha=0.2)
    axis.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(OUTPUT / "error_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Failure galleries

Each gallery uses the same IDs for every model. Categories separate high-DAB underestimation,
false-positive staining, blur, morphology mismatch and poor pair alignment.


In [ ]:
def select_ids(category, count=GALLERY_SIZE):
    pivot = errors.pivot_table(index="id", columns="model", values=category, aggfunc="mean")
    return pivot.mean(axis=1).sort_values(ascending=False).head(count).index.tolist()


errors["high_dab_underestimate"] = np.maximum(-errors["dab_fraction_signed_error"], 0) * errors["real_dab_fraction"]
errors["false_positive_dab"] = np.maximum(errors["dab_fraction_signed_error"], 0) * (1 - errors["real_dab_fraction"])
errors["blur_error"] = np.maximum(1 - errors["sharpness_ratio"], 0)
errors["morphology_error"] = 1 - errors["pred_real_nuclear_ssim"]
errors["pair_alignment_problem"] = 1 - errors["nuclear_ssim_aligned"]

categories = ["high_dab_underestimate", "false_positive_dab", "blur_error",
              "morphology_error", "pair_alignment_problem"]
record_by_id = {record["id"]: record for record in records}

for category in categories:
    identities = select_ids(category)
    columns = ["H&E", "Real HER2"] + list(models)
    fig, axes = plt.subplots(len(identities), len(columns),
                             figsize=(3 * len(columns), 3 * len(identities)), squeeze=False)
    for row, identity in enumerate(identities):
        record = record_by_id[identity]
        paths = [DATA_ROOT / record["he"], DATA_ROOT / record["ihc"]]
        paths += [root / f"{identity}.png" for root in models.values()]
        for column, (title, path) in enumerate(zip(columns, paths)):
            axes[row, column].imshow(Image.open(path).convert("RGB"))
            axes[row, column].set_title(title if row == 0 else "")
            axes[row, column].axis("off")
        axes[row, 0].set_ylabel(identity.split("/")[-1], fontsize=8)
    fig.suptitle(category.replace("_", " ").title())
    fig.tight_layout()
    fig.savefig(OUTPUT / f"gallery_{category}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 7. Inventory qualitative experiment galleries

These images may come from selected validation examples and are displayed separately. Their small
metrics files are not treated as full-test results.


In [ ]:
example_files = []
for root in SEARCH_ROOTS:
    for path in root.rglob("example-*.png"):
        if not any(path.is_relative_to(model_root) for model_root in models.values()):
            example_files.append(path)
example_files = sorted(set(example_files))

gallery_inventory = pd.DataFrame([
    {"experiment": path.parent.name, "file": path.name, "path": str(path)}
    for path in example_files
])
gallery_inventory.to_csv(OUTPUT / "qualitative_experiment_inventory.csv", index=False)
display(gallery_inventory)

for experiment, frame in gallery_inventory.groupby("experiment"):
    paths = [Path(path) for path in frame.path.head(4)]
    fig, axes = plt.subplots(1, len(paths), figsize=(7 * len(paths), 5), squeeze=False)
    for axis, path in zip(axes[0], paths):
        axis.imshow(Image.open(path).convert("RGB"))
        axis.set_title(path.name)
        axis.axis("off")
    fig.suptitle(experiment)
    fig.tight_layout()
    fig.savefig(OUTPUT / f"qualitative_{experiment}.png", dpi=140, bbox_inches="tight")
    plt.show()


## 8. Evidence summary and next-experiment flags

The generated flags identify which hypotheses are supported by this audit. Final experimental
decisions should use the exported tables and galleries, not the flags alone.


In [ ]:
flags = {}
for model, frame in errors.groupby("model"):
    rho_align = spearmanr(frame["nuclear_ssim_aligned"], frame["ssim"], nan_policy="omit").statistic
    rho_dab = spearmanr(frame["real_dab_fraction"], frame["dab_fraction_abs_error"], nan_policy="omit").statistic
    high = frame[frame.dab_stratum == "high"]
    flags[model] = {
        "alignment_association": float(rho_align),
        "dab_error_association": float(rho_dab),
        "mean_high_dab_signed_error": float(high.dab_fraction_signed_error.mean()),
        "median_sharpness_ratio": float(frame.sharpness_ratio.median()),
        "candidate_alignment_fix": bool(rho_align > 0.20),
        "candidate_dab_balancing": bool(rho_dab > 0.35 or high.dab_fraction_signed_error.mean() < -0.05),
        "candidate_sharpness_or_perceptual_loss": bool(frame.sharpness_ratio.median() < 0.70),
    }

(OUTPUT / "diagnostic_flags.json").write_text(json.dumps(flags, indent=2))
print(json.dumps(flags, indent=2))

recommendation_rows = []
for model, values in flags.items():
    if values["candidate_alignment_fix"]:
        recommendation_rows.append({
            "model": model, "evidence": "SSIM increases with pair-alignment quality",
            "next_test": "alignment-aware filtering/weighting or shift-tolerant perceptual loss",
        })
    if values["candidate_dab_balancing"]:
        recommendation_rows.append({
            "model": model, "evidence": "error increases with real DAB or strong DAB is underestimated",
            "next_test": "DAB-stratified sampling plus stain-intensity auxiliary loss",
        })
    if values["candidate_sharpness_or_perceptual_loss"]:
        recommendation_rows.append({
            "model": model, "evidence": "predictions are systematically less sharp than real IHC",
            "next_test": "calibrated LPIPS/feature loss while keeping the same split and sampler",
        })
recommendations = pd.DataFrame(recommendation_rows)
recommendations.to_csv(OUTPUT / "next_experiment_hypotheses.csv", index=False)
display(recommendations)


In [ ]:
report = {
    "models": {label: str(root) for label, root in models.items()},
    "test_count": len(records),
    "audit_size": AUDIT_SIZE,
    "dab_threshold": DAB_THRESHOLD,
    "notes": [
        "Alignment scores are cross-stain morphology proxies, not registration ground truth.",
        "DAB groups are data-driven strata, not clinical 0/1+/2+/3+ labels.",
        "Full-test metrics remain primary; selected galleries are diagnostic only.",
    ],
}
(OUTPUT / "audit_report.json").write_text(json.dumps(report, indent=2))

archive_path = WORK_ROOT / "mist-multimodel-error-audit.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT.parent))
print("Saved:", archive_path, f"({archive_path.stat().st_size / 1024**2:.1f} MB)")

from IPython.display import FileLink, display
display(FileLink(str(archive_path)))
